# 05 · Language signals, temporal replay and leakage
### BeliefLens · STAC AI Workshop · NYC · 21 October 2026

## Learning objectives
1. Define a structured financial signal
2. Use only information available at decision time
3. Detect look-ahead and vintage leakage
4. Compare performance after costs

All companies, numbers, labels and market observations in these notebooks are synthetic teaching examples. Offline outputs come from explicit deterministic rules, not an LLM. The same JavaScript case engine runs in the browser and through this Python client. Only a cell explicitly using `live=True` calls Together AI. No financial action is executed.

**Requirements:** Python 3.10+, Node.js 20+, Jupyter. No model weights or API key are needed for the default exercises. Run cells in order.

In [ ]:
from pathlib import Path
import sys, json, math, random, statistics
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "lab_client.py").exists()), None)
if root is None:
    raise RuntimeError("Extract the complete workshop ZIP and open Jupyter inside its folder.")
if str(root) not in sys.path: sys.path.insert(0, str(root))
from lab_client import run, compact, save
print("Shared engine ready:", compact(run()))

## A deliberately controlled market experiment
Generate a dated news score and a subsequent synthetic return. The valid signal uses the news score; the invalid signal uses the sign of tomorrow’s return. This shows leakage mechanics, not trading performance. A real language signal would come from a versioned model run on eligible text and be evaluated with the same timing discipline.

In [ ]:
rng=random.Random(7)
market=[]
for t in range(160):
    news_score=rng.uniform(-1,1)
    future_return=0.001*news_score+rng.gauss(0,0.012)
    market.append({"day":t,"news_score":news_score,"return_next":future_return})
held_out=market[100:]
def strategy(data, leaked=False):
    previous=0; pnl=[]
    for row in data:
        score=row["return_next"] if leaked else row["news_score"]
        position=1 if score>=0 else -1
        cost=0.0005*abs(position-previous)
        pnl.append(position*row["return_next"]-cost)
        previous=position
    return {"mean_daily_pnl":statistics.mean(pnl),"total_uncompounded":sum(pnl),"days":len(pnl)}
print("Valid held-out signal:",strategy(held_out))
print("Leaked future-return signal:",strategy(held_out,True))
print("Cash baseline: 0 (interest ignored in this synthetic example)")
assert strategy(held_out,True)["mean_daily_pnl"] > strategy(held_out)["mean_daily_pnl"]

## Vintage eligibility
A revision may refer to the same economic period while becoming available later. You must join by availability, not just by the period named in a record.

In [ ]:
vintages=[{"period":"Q2","available_day":10,"value":20},{"period":"Q2","available_day":20,"value":17}]
def as_of_value(day):
    available=[v for v in vintages if v["available_day"]<=day]
    return max(available,key=lambda v:v["available_day"])["value"] if available else None
print("At day 15:",as_of_value(15),"At day 25:",as_of_value(25))
assert as_of_value(15)==20
print("Shared gate under deliberate future leakage:",compact(run({"futureLeak":True})))

## Learning check
Does a higher leaked backtest score mean the model anticipated stress? **No.** The experiment gave the strategy information it could not have had.

**Extend the test:** record model version, prompt, source snapshot, decision time, availability time and transaction-cost assumptions. Use chronological validation and quantify sampling uncertainty. More conservative risk numbers alone do not prove earlier risk detection.

## What you have demonstrated
- **Define a structured financial signal:** Converted a numerical news score to a reproducible position rule
- **Use only information available at decision time:** Separated score from future return
- **Detect look-ahead and vintage leakage:** Demonstrated an invalid backtest and a vintage-aware join
- **Compare performance after costs:** Included turnover costs and a cash baseline

**Exit ticket:** State one result, the evidence supporting it, and one limit that prevents you from claiming more. Save a run JSON or your numerical results as the experiment record.